In [72]:
import pandas as pd
import numpy as np
import pandapipes as pp

In [73]:
class Simulation():
    def __init__(self,DHN,fluid="water"):
        self.DHN=DHN
        self.fluid=fluid
    def Network_from_Graph_to_Pandapipes(self,DHN):
        """This function will create a pandapipes network from a graph created by the class DHN"""
        self.net = pp.create_empty_network(fluid="water")
        for node in self.DHN.graph.nodes:
            pp.create_junction(self.net, pn_bar=1, tfluid_k=20, name=node)
            if self.DHN.graph.nodes[node]['type']=="plant":
                pp.create_ext_grid(self.net, junction=node, p_bar=1.1, t_k=20)
                pp.create_sink(self.net, junction=node, mdot_kg_per_s=0.1)
        for edge in self.DHN.graph.edges(data=True):
            pp.create_pipe(self.net, from_junction=edge[0], to_junction=edge[1], length=edge[2]['length'], diameter=0.1, name=str(edge[0])+"-"+str(edge[1]))

In [74]:
import pandapipes as pp

# 1. Crear red vacía
net = pp.create_empty_network(fluid="water")

# =============================================================================
# NUDOS (JUNCTIONS)
# =============================================================================
# Nota: Las temperaturas en pandapipes se introducen en Kelvin (K = °C + 273.15)
# Suponemos Ida a 80 °C (353.15 K) y Retorno esperado a 60 °C (333.15 K)

# Nudos de la Central / Fuente
j_fuente_ida = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Fuente_Ida")
j_fuente_ret = pp.create_junction(net, pn_bar=1.5, tfluid_k=333.15, name="Fuente_Retorno")

# Nudos en casa del Consumidor
j_cons_ida = pp.create_junction(net, pn_bar=4.8, tfluid_k=353.15, name="Consumidor_Ida")
j_cons_ret = pp.create_junction(net, pn_bar=1.7, tfluid_k=333.15, name="Consumidor_Retorno")


# =============================================================================
# TUBERÍAS (PIPES) - IDA Y RETORNO SIN PÉRDIDAS TÉRMICAS (k_w = 0)
# =============================================================================

pipe_ida = pp.create_pipe_from_parameters(
    net, from_junction=j_fuente_ida, to_junction=j_cons_ida,
    length_km=0.1616, inner_diameter_mm=150, u_w_per_m2k=0.35/ (np.pi * 0.15), k_mm=0.1*1000, text_k=273.2, name="Tubo_Ida"
)

pipe_retorno = pp.create_pipe_from_parameters(
    net, from_junction=j_cons_ret, to_junction=j_fuente_ret,
    length_km=0.1616, inner_diameter_mm=150, u_w_per_m2k=0.35 / (np.pi * 0.15), k_mm=0.1*1000, text_k=273.2, name="Tubo_Retorno"
)


# =============================================================================
# COMPONENTE CONSUMIDOR (Potencia en W y Temp de Retorno)
# =============================================================================
# Conectamos el nudo de ida con el de retorno. 
# El sistema extraerá agua de la ida, absorberá los Watts indicados y la devolverá al retorno a la t_return_k.

pp.create_heat_consumer(
    net,
    from_junction=j_cons_ida,
    to_junction=j_cons_ret,
    qext_w=150000,
    treturn_k=318,
    name="Consumidor"
)
# =============================================================================
# CONDICIONES DE CONTORNO (EXT_GRID)
# =============================================================================
# Fijamos la presión y temperatura de salida en la ida
pp.create_circ_pump_const_pressure(net,flow_junction=j_fuente_ida,return_junction=j_fuente_ret,p_flow_bar=3,plift_bar=0.5,t_flow_k=353 ,name='Grid'
)





# =============================================================================
# SIMULACIÓN Y RESULTADOS
# =============================================================================
pp.pipeflow(net, mode="bidirectional")

In [75]:
net.res_pipe

,v_mean_m_per_s,p_from_bar,p_to_bar,t_from_k,t_to_k,t_outlet_k,mdot_from_kg_per_s,mdot_to_kg_per_s,vdot_m3_per_s,reynolds,lambda,dp_friction_loss_bar
0,0.061343,3.000000,2.991085,353.0,351.985860,351.985860,1.053771,-1.053771,0.001084,25094.941641,0.452419,0.008915
1,0.060214,2.508784,2.500000,318.0,317.428484,317.428484,1.053771,-1.053771,0.001064,14926.282536,0.454156,0.008784


In [76]:
net.res_circ_pump_pressure

,p_from_bar,p_to_bar,t_from_k,t_to_k,t_outlet_k,mdot_from_kg_per_s,mdot_to_kg_per_s,vdot_m3_per_s,deltat_k,qext_w
0,2.5,3.0,317.428484,353.0,353.0,1.053771,-1.053771,0.001074,-35.571516,157009.1526


In [77]:
net.res_heat_consumer

,p_from_bar,p_to_bar,t_from_k,t_to_k,t_outlet_k,mdot_from_kg_per_s,mdot_to_kg_per_s,vdot_m3_per_s,deltat_k,qext_w
0,2.991085,2.508784,351.98586,318.0,318.0,1.053771,-1.053771,0.001074,33.98586,150000.0


In [78]:
net.res_junction

,p_bar,t_k
0,3.000000,353.000000
1,2.500000,317.428484
2,2.991085,351.985860
3,2.508784,318.000000
